# Phase 2: 기본 쿼리
**DB:** `data/franchise.db` (맛나국밥 가상 프랜차이즈)  
**테이블:** stores / menu_items / daily_sales / promotions / inventory

| 단계 | 내용 |
|------|------|
| 1 | 단일 테이블 — SELECT / WHERE / ORDER BY |
| 2 | GROUP BY 집계 |
| 3 | 2테이블 JOIN |
| 4 | 3테이블 JOIN + GROUP BY |

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/franchise.db')

def query(sql):
    return pd.read_sql_query(sql, conn)

---
## 1. 단일 테이블 조회
### 과제 1-1
`stores` 테이블에서 **가맹점만** 조회하되,  
`store_name`, `region`, `store_type`, `seating_capacity`만 출력하고  
`seating_capacity` 기준 **내림차순** 정렬

In [2]:
query("""
SELECT store_name, region, store_type, seating_capacity
FROM stores
WHERE store_type = '가맹'
ORDER BY seating_capacity DESC
""")

,store_name,region,store_type,seating_capacity
0,맛나국밥 송파구점,서울,가맹,60
1,맛나국밥 서초구점,서울,가맹,60
2,맛나국밥 노원구점,서울,가맹,60
3,맛나국밥 수원시점,경기,가맹,60
4,맛나국밥 고양시점,경기,가맹,60
5,맛나국밥 남동구점,인천,가맹,60
6,맛나국밥 부산진구점,부산,가맹,60
7,맛나국밥 강남구점,서울,가맹,50
8,맛나국밥 마포구점,서울,가맹,50
9,맛나국밥 종로구점,서울,가맹,50


### 과제 1-2
`menu_items` 테이블에서 **가격이 10,000원 이상인 메뉴**만 조회하되,  
`menu_name`, `category`, `price`, `cost`를 출력하고  
`price` 기준 **오름차순** 정렬

In [3]:
query("""
SELECT menu_name, category, price, cost
FROM menu_items
WHERE price >= 10000
ORDER BY price ASC
""")

,menu_name,category,price,cost
0,설렁탕,탕,11000,4500
1,갈비탕,탕,14000,6000
2,도가니탕,탕,15000,6500
3,수육,안주,28000,11000


---
## 2. GROUP BY 집계
### 과제 2-1
`daily_sales` 테이블에서 **매장(`store_id`)별 총 매출액 합계**를 구하고  
총매출 기준 **내림차순** 정렬

In [4]:
query("""
SELECT store_id, SUM(total_amount) AS 총매출
FROM daily_sales
GROUP BY store_id
ORDER BY 총매출 DESC
""")

,store_id,총매출
0,5,654547000
1,12,629635000
2,11,620811000
3,1,604537000
4,19,601874000
5,6,589305000
6,8,580658000
7,3,557547000
8,15,535210000
9,7,521674000


### 과제 2-2
`daily_sales` 테이블에서 **메뉴(`menu_id`)별 총 판매수량과 총 매출액**을 구하고  
총수량 기준 **내림차순** 정렬

In [5]:
query("""
SELECT menu_id, SUM(quantity) AS 총수량, SUM(total_amount) AS 총매출
FROM daily_sales
GROUP BY menu_id
ORDER BY 총수량 DESC
""")

,menu_id,총수량,총매출
0,3,175205,2628075000
1,2,174782,2446948000
2,1,174538,1919918000
3,6,151824,151824000
4,5,151574,1061018000
5,8,87512,525072000
6,7,86842,434210000
7,4,50559,1415652000


### 과제 2-3
`daily_sales` 테이블에서 **월별 총 매출액**을 구하고  
월 기준 **오름차순** 정렬  
> `SUBSTR(sale_date, 1, 7)`로 `'2025-01'` 형태 추출

In [6]:
query("""
SELECT SUBSTR(sale_date, 1, 7) AS 월, SUM(total_amount) AS 총매출
FROM daily_sales
GROUP BY 월
ORDER BY 월 ASC
""")

,월,총매출
0,2025-01,1009678000
1,2025-02,922877000
2,2025-03,883271000
3,2025-04,846594000
4,2025-05,877734000
5,2025-06,852966000
6,2025-07,796570000
7,2025-08,812976000
8,2025-09,842980000
9,2025-10,875139000


---
## 3. 2테이블 JOIN
### 과제 3-1
`daily_sales` × `stores` JOIN으로  
**매장 이름(`store_name`)별 총 매출액**을 구하고 총매출 내림차순 정렬  
> 연결 키: `daily_sales.store_id = stores.store_id`

In [7]:
query("""
SELECT store_name, SUM(total_amount) AS 총매출
FROM daily_sales
JOIN stores ON daily_sales.store_id = stores.store_id
GROUP BY store_name
ORDER BY 총매출 DESC
""")

,store_name,총매출
0,맛나국밥 영등포구점,654547000
1,맛나국밥 용인시점,629635000
2,맛나국밥 고양시점,620811000
3,맛나국밥 강남구점,604537000
4,맛나국밥 달서구점,601874000
5,맛나국밥 강서구점,589305000
6,맛나국밥 노원구점,580658000
7,맛나국밥 종로구점,557547000
8,맛나국밥 해운대구점,535210000
9,맛나국밥 서초구점,521674000


### 과제 3-2
`daily_sales` × `menu_items` JOIN으로  
**메뉴 이름(`menu_name`)별 총 판매수량과 총 매출액**을 구하고 총매출 내림차순 정렬  
> 연결 키: `daily_sales.menu_id = menu_items.menu_id`

In [8]:
query("""
SELECT menu_name, category, SUM(quantity) AS 총수량, SUM(total_amount) AS 총매출
FROM daily_sales
JOIN menu_items ON daily_sales.menu_id = menu_items.menu_id
GROUP BY menu_name
ORDER BY 총매출 DESC
""")

,menu_name,category,총수량,총매출
0,도가니탕,탕,175205,2628075000
1,갈비탕,탕,174782,2446948000
2,설렁탕,탕,174538,1919918000
3,수육,안주,50559,1415652000
4,만두,사이드,151574,1061018000
5,맥주,주류,87512,525072000
6,소주,주류,86842,434210000
7,공기밥,사이드,151824,151824000


---
## 4. 3테이블 JOIN + GROUP BY
### 과제 4-1
`daily_sales` × `stores` × `menu_items` 3테이블 JOIN으로  
**지역(`region`)별, 카테고리(`category`)별 총 매출액**을 구하고 총매출 내림차순 정렬  
> JOIN을 두 번 연결: `store_id` → `stores`, `menu_id` → `menu_items`

In [9]:
query("""
SELECT region, category, SUM(total_amount) AS 총매출
FROM daily_sales
JOIN stores     ON daily_sales.store_id = stores.store_id
JOIN menu_items ON daily_sales.menu_id  = menu_items.menu_id
GROUP BY region, category
ORDER BY 총매출 DESC
""")

,region,category,총매출
0,서울,탕,2974345000
1,경기,탕,1403584000
2,대구,탕,1066415000
3,부산,탕,959825000
4,서울,안주,602980000
5,인천,탕,590772000
6,서울,사이드,515021000
7,서울,주류,406628000
8,경기,안주,282660000
9,경기,사이드,244646000


In [10]:
conn.close()